# 課題③ HuggingFace で LLM 推論パラメータを探索する

**`# TODO:` のセルを自分で完成させてください。**  
わからない場合は `solutions/sol_03_llm_inference.ipynb` を参照してください。

---

## セットアップ

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from collections import Counter
from itertools import combinations
from transformers import AutoTokenizer, AutoModelForCausalLM

if os.path.exists('/data/shared/models'):
    os.environ['HF_HOME'] = '/data/shared/models'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用デバイス: {device}')
torch.manual_seed(42)

---

## Step 1: モデルの読み込み（完成済み）

In [ ]:
MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print('モデルロード完了')

---

## Step 2: generate_text() 関数の実装

以下の仕様で `generate_text()` 関数を完成させてください。

- `tokenizer()` でプロンプトをテンソルに変換し、`device` へ移動する
- `model.generate()` を呼び出す（`do_sample`, `temperature`, `top_p`, `repetition_penalty`, `max_new_tokens` を渡す）
- `torch.no_grad()` ブロック内で実行する
- プロンプト部分のトークンを除いた生成テキストだけを返す

In [ ]:
def build_prompt(user_message: str, system: str = 'You are a helpful assistant.') -> str:
    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': user_message},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate_text(
    prompt: str,
    max_new_tokens: int = 150,
    do_sample: bool = True,
    temperature: float = 0.7,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
) -> str:
    """プロンプトを受け取り、生成テキスト（プロンプト部分を除く）を返す"""

    # TODO: プロンプトをトークナイズして device に移動してください
    inputs = None
    input_len = None

    # TODO: torch.no_grad() 内で model.generate() を呼び出してください
    # 引数: max_new_tokens, do_sample, temperature, top_p,
    #       repetition_penalty, pad_token_id=tokenizer.eos_token_id
    outputs = None

    # TODO: プロンプト部分を除いたトークンをデコードして返してください
    return None


# 確認
question = '深層学習を一言で説明してください。'
response = generate_text(build_prompt(question))
assert isinstance(response, str) and len(response) > 0, '空の文字列が返されました'
print(f'generate_text() ✓')
print(f'質問: {question}')
print(f'回答: {response}')

---

## Step 3: Temperature の実験

`temperatures = [0.1, 0.7, 1.3]` の 3 値で同一プロンプトを生成し、  
各出力の Type-Token Ratio（TTR）を棒グラフで可視化してください。

$$\text{TTR} = \frac{\text{ユニークトークン数}}{\text{総トークン数}}$$

In [ ]:
def type_token_ratio(text: str) -> float:
    """テキストの Type-Token Ratio を返す"""
    # TODO: tokenizer.encode() でトークンIDリストを得て TTR を計算してください
    pass


prompt_temp  = build_prompt('AIの未来について、自由に語ってください。')
temperatures = [0.1, 0.7, 1.3]
colors       = ['#0D4A38', '#7C5C00', '#991B1B']
temp_results = {}

# TODO: temperatures をループし、各 temperature で generate_text() を呼び出して
#       temp_results[t] = response に格納してください（シードは torch.manual_seed(42) で固定）


# TODO: temp_results から TTR を計算し、棒グラフで可視化してください


---

## Step 4: Top-p の実験

`temperature=0.9` で固定し、`top_p = [0.5, 0.9, 1.0]` の 3 値で比較してください。  
各出力を表示するだけで構いません。

In [ ]:
prompt_topp = build_prompt('料理のレシピを1つ考えてください。')
top_ps      = [0.5, 0.9, 1.0]

# TODO: top_ps をループし、各 top_p で generate_text() を呼び出して結果を表示してください


---

## Step 5: 複数回生成のばらつきを可視化

`temperature = [0.1, 0.7, 1.3]` それぞれで 5 回生成し、  
全ペアの Jaccard 類似度の平均を棒グラフで描いてください。

$$\text{Jaccard}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

（高いほど出力が似ている ＝ 多様性が低い）

In [ ]:
def token_overlap(text_a: str, text_b: str) -> float:
    """2テキスト間の Jaccard 類似度（トークン集合ベース）を返す"""
    # TODO: tokenizer.encode() でトークンIDセットを得て Jaccard を計算してください
    pass


N_TRIALS    = 5
prompt_var  = build_prompt('今日の気分を一文で表現してください。')
test_temps  = [0.1, 0.7, 1.3]
colors_var  = ['#0D4A38', '#7C5C00', '#991B1B']

# TODO: 各 temperature で N_TRIALS 回生成し、全ペアの Jaccard 平均を計算して
#       棒グラフで可視化してください
#       （シードは torch.manual_seed(i) で各試行番号ごとに変える）


---

## チャレンジ問題

`repetition_penalty = [1.0, 1.2, 1.5]` の 3 値で長めの出力（`max_new_tokens=150`）を生成し、  
各出力の **2-gram 重複率** を計算して比較してください。

$$\text{2-gram重複率} = 1 - \frac{\text{ユニーク2-gram数}}{\text{総2-gram数}}$$

In [ ]:
# TODO: repetition_penalty = [1.0, 1.2, 1.5] で生成し、2-gram 重複率を比較してください
